# Day 8: Autonomous Coding Agents (Claude Code / Devin)

## The Problem

Simple bug fixes can be done with a single LLM call plus one tool call. Autonomous coding agents exist for a different class of problem: multi file, multi step, long horizon engineering tasks such as "refactor auth module across 15 files" or "investigate an intermittent bug."

These tasks need three things a simple system does not have:
1. State maintained across many tool calls
2. Self verification of the agent's own work
3. Human checkpoints before high stakes actions, not after

## The Architecture (Claude Code, publicly documented)

```
User prompt
   |
Plan Mode (read only tools: Read, Grep, Glob, no edits allowed)
   |
Human reviews and approves the plan
   |
Execution loop (Edit, Write, Bash, MCP tools; agent re plans after every tool result)
   | subagents can be spawned for isolated exploration, only a summary returns to main context
   |
Verification (agent runs tests, fixes failures)
   |
Output: diff or PR or summary
```

Note: Devin's internal architecture is not publicly documented in the same depth. It likely follows a similar plan then execute pattern (sandboxed VM, interruptible plan), but this is inference, not a verified fact.

## Why Separate Planning from Execution (core cross question)

If the agent improvises step by step without a distinct plan phase:
- A wrong direction early on can corrupt every subsequent step, and by the time a human notices, many bad edits already exist
- There is no single artifact to review before damage is done
- Ambiguous intent only becomes visible after code has already changed

Separating planning from execution creates a checkpoint where a human corrects direction cheaply, before wasted execution, instead of correcting a half finished diff afterward.

## Key Tradeoffs

- Autonomy vs control: Plan Mode adds overhead for simple tasks but prevents costly mistakes on large or high stakes changes
- Context cost vs completeness: subagents keep the main context clean, but their summaries can lose nuance
- Latency vs correctness: a verification loop (run tests, fix, repeat) costs time but reduces the rate of silently broken code being shipped
- Determinism vs flexibility: an agentic loop (re plan after each tool result) adapts better than a fixed script, but is harder to debug when it fails

## Failure Modes

- Hallucination: mitigated by grounding, forcing the agent to Read or Grep real files before editing
- Ambiguous intent: surfaced during the plan review step instead of discovered mid execution
- Tool failure: the agent sees the error as tool output and re plans, which is why execution is a loop and not a fixed script
- Adversarial input: permission modes require explicit approval for risky actions such as destructive git operations or commands touching files outside the working directory

## Checkpointing and Memory (in simple terms)

Checkpointing means the agent saves its progress to disk during a long task, similar to a save point in a video game.

Why it matters: if a long task (20 minutes, 40+ tool calls) is interrupted by a crash, lost connection, or the user closing the terminal, the agent should not have to start over. Without checkpointing, all progress and context is lost, and the agent would re explore files it had already understood.

In Claude Code this is exposed as:
- `claude --continue` or `claude --resume` to resume a previous session from where it left off
- `claude --worktree` to run parallel sessions on separate git branches, so multiple agents do not collide with each other's state

Analogy: like persisting a visited set in a DFS traversal so that if the program crashes, you do not have to revisit already processed nodes on restart.

## Self Verification: Why It Is Harder Than It Looks

Naive idea: define the test criteria during the plan phase and instruct the model not to break or ignore that test.

Problem: instructions alone are not reliable. The model may still weaken an assertion, skip a test, or misinterpret a flaky test as a pass, because the same agent is both executor and grader (a self grading problem).

### Solution 1: Independent Judge

Use a separate LLM instance as a judge, with its own context, given only the diff, the original test file, and the task spec. It should not see the execution history or reasoning trail, so it has no narrative bias toward declaring success.

### Solution 2: Escalation to a Human, but Conditionally

Escalating every failure to a human defeats the purpose of automation. Escalation should be triggered by structural, code level signals rather than by asking the LLM whether it is unsure. Important nuance: even the judge LLM can be wrong, so the decision to escalate should not depend solely on the judge's opinion.

### Concrete Escalation Triggers

Structural triggers (detected by diff or code analysis, no LLM judgment needed):
1. Test file modified in the same diff as the source code being tested
2. Assertion weakened (for example assertEquals changed to assertTrue, or a tolerance value increased)
3. Test skipped or disabled (skip annotations, commented out assertions)
4. Test count decreased compared to before the change

Scope triggers (deterministic comparison):
5. Files touched outside the scope stated in the original plan
6. Destructive operations present in the diff (for example git reset hard, DROP TABLE, rm rf, force push)

Confidence based triggers (where the LLM judge is involved, but only as a signal, not the final decision):
7. Disagreement between executor and judge on an objective result (for example executor says tests pass, judge independently runs them and gets a failing exit code)
8. Judge outputs a structured confidence score, and a score below a fixed threshold routes to human review

### Core Principle

The LLM should propose (judgments, confidence scores, verdicts). A deterministic rule layer in code should dispose (decide whether to escalate to a human or auto merge). This way, the final escalation decision cannot be bypassed by a hallucination or a prompt injection, because it does not live inside a prompt, it lives in code.

## Claude Code vs Cursor vs Codex: Same Category, Different Design Philosophy

All three sit in the autonomous coding agent category, but they represent different paradigms of deployment and autonomy, not different underlying problems.

- Codex says "tell me what to do and I will handle it autonomously"
- Cursor says "code alongside me in a familiar environment"
- Claude Code says "let us reason through this together in the terminal"

### Cursor: IDE first
A full fork of VS Code rebuilt around an AI first workflow. You work inside it all day rather than invoking it occasionally. It is model agnostic: you can choose between models from OpenAI, Anthropic, Google, and xAI natively, with no workaround needed. Strength: tab completion, inline edits, visual diff review, staying in the driver's seat.

### Codex: cloud first, async
Runs tasks in isolated cloud containers. It clones the repo into a temporary cloud VM, does all edits and test runs on that copy, and the local machine is never touched during execution. Output comes back as a diff or a pull request that the developer reviews and merges (locally, via git pull, or through the GitHub UI). This is why Codex fits a batch or async workflow: you assign a task, walk away, and come back to a finished result, and you can fire off several such tasks in parallel since they run in separate containers.

### Claude Code: terminal native, interactive
Runs locally in the terminal and edits real files on disk directly, no clone or sync back step. This is a synchronous, two way interaction: you give a command, see the result, and respond immediately, similar to pair programming. This immediacy is what makes it feel interactive rather than batch.

### Interactive dialogue vs batch and parallel workloads, defined simply

- Interactive terminal dialogue: real time, synchronous back and forth, like a phone call
- Batch or async workload: assign a task and walk away, like sending an email and checking back later for the reply
- Parallel workload: firing off multiple such async tasks at once, each running independently, like emailing several people at once and waiting for all replies together

## Correcting a Common Misconception: Is Claude Code Open Source and Model Agnostic?

Partially true, and this distinction matters. The Claude Code CLI harness itself is open source, but by default it is locked to Anthropic's own model family (Claude Opus, Sonnet, Haiku). Using non Anthropic models with it requires a compatibility gateway layer (for example LiteLLM or Bifrost) rather than native, first class support. Codex has the same kind of lock in on the OpenAI side: it runs specifically on the GPT-5.x Codex model family by design. Cursor is the one that is genuinely model agnostic natively, without needing a gateway workaround.

So: open source code does not imply open model choice. Claude Code's openness is about the harness (the orchestration code), not about which model it talks to by default.

## Token Efficiency Comparison (important interview stat, verify current numbers before quoting)

Independent testing found Claude Code used 5.5x fewer tokens than Cursor for an identical benchmark task (33K tokens for Claude Code vs 188K tokens for Cursor's agent, with Cursor also hitting errors). Codex, in turn, is reported as 2 to 4x more token efficient than Claude Code, particularly for batch workloads.

Why the difference exists, even though all three converge on the same high level plan, execute, verify pattern:

1. Context management strategy: Claude Code aggressively prunes and summarizes context returned from subagents rather than carrying everything forward, which is a large part of why it beats Cursor on token count for the same task
2. Model level tool calling precision: a model trained specifically to be efficient and accurate at tool use will need fewer retries and clarifying loops, independent of the orchestration layer around it
3. Deployment goal shapes the incentive: Codex is optimized for cloud batch throughput, so minimizing per task token cost is a first class design goal; Claude Code is optimized for depth of reasoning in an interactive single session, where thoroughness is prioritized over raw token minimization

Conclusion for interviews: assuming "the architecture is the same, so efficiency differences must be pure orchestration" is an oversimplification. The high level pattern (plan, execute, verify) has converged across tools by 2026, but token efficiency differences come from a combination of context pruning strategy, the model's own tool calling precision, and the product's deployment goal (batch throughput vs interactive depth), not from architecture alone.

## Sandbox Isolation and the Real Meaning of "Safer"

A natural question: if Codex runs in a cloud sandbox, is it safer than Claude Code, which edits real local files directly?

The honest answer is that sandboxing changes the type of risk, it does not simply reduce risk overall.

What sandboxing does give Codex: the local machine is never touched, so a bad edit only corrupts a disposable cloud copy, not the developer's real filesystem.

What this does not account for:

1. Claude Code is not as exposed as it first sounds, because it almost always operates inside a git repository. Every edit is reversible through git diff, git checkout, or git reset, and Plan Mode adds a read only phase before any edit happens at all. Git itself acts as a safety net.
2. Because Codex is async, review happens later and in bulk, on a large finished diff, rather than in real time. Delayed, batch style review has a documented downside: analyses have found sharp increases in incidents per pull request and in PRs merged without review when tools over engineer changes beyond what was asked and nobody catches it in time.
3. So the real tradeoff is: Claude Code produces small, frequent, immediately visible changes (easy to catch a mistake right away, protected by git), while Codex produces large, delayed, batch visible changes (protected from touching your live disk, but the mistake can hide until the diff is actually reviewed).

Correct framing for interviews: "safer" is not a single axis. There are two separate risks:
- Immediate local damage risk: lower for Codex (sandboxed), higher for Claude Code if git discipline is weak
- Review discipline risk: higher for Codex (async, batch review encourages skimming), lower for Claude Code (real time, mistakes are caught as they happen)

In both cases, the actual safety net is the human review process, not the architecture itself. Sandbox isolation reduces local machine risk, but it does not remove the need for careful review, it just moves where and when that review has to happen.

## Interview Ready Talking Points

- Be able to draw the plan then execute then verify diagram from memory
- Be able to explain the cross question answer in two sentences: separating planning from execution creates a cheap correction point before execution, preventing early mistakes from compounding
- Be able to name at least 3 concrete escalation triggers that are structural rather than instruction based
- Be able to explain why "LLM proposes, deterministic rules dispose" is safer than trusting a judge LLM's own escalation decision
- Be able to explain checkpointing with the DFS visited set analogy
- Know the honest boundary: Claude Code's plan mode and subagent isolation are publicly documented; Devin's internals are inferred by analogy, not confirmed